In [1]:
#Step 1: Import necessary libraries and load the dataset
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Load the FYP dataset
df = pd.read_csv('fyp_runner_dataset.csv')
print(f"Dataset loaded successfully with {df.shape[0]} rows and {df.shape[1]} columns.")

Dataset loaded successfully with 1000 rows and 16 columns.


In [2]:
#Step 2: Preprocess the data
# Map the target_race to a numerical distance in kilometers
distance_mapping = {
    '5K': 5.0,
    '10K': 10.0,
    'Half Marathon': 21.0975,
    'Full Marathon': 42.195
}
df['target_distance_km'] = df['target_race'].map(distance_mapping)

# Map gender to binary (Male = 1, Female = 0)
df['gender_encoded'] = df['gender'].map({'Male': 1, 'Female': 0})

# Drop columns that are not useful for prediction (like runner_id) or have been replaced
df_clean = df.drop(columns=['runner_id', 'gender', 'target_race'])

print("Preprocessing complete. Data is formatted for ML.")

Preprocessing complete. Data is formatted for ML.


In [3]:
# Step 3: Define features and target variable
# Define features (X) and target variable (y)
features = [
    'age', 'gender_encoded', 'resting_hr', 'max_hr', 'vo2_max', 
    'weekly_mileage_km', 'avg_easy_pace_min_km', 'avg_tempo_pace_min_km', 
    'training_consistency', 'pb_5k_mins', 'pb_10k_mins', 
    'pb_half_mins', 'pb_full_mins', 'target_distance_km'
]
target = 'actual_finish_time_mins'

X = df_clean[features]
y = df_clean[target]

print(f"Number of features selected: {len(features)}")

Number of features selected: 14


In [4]:
#Step 4: Train the models
# 80/20 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize models
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)

# Train the models
rf_model.fit(X_train, y_train)
xgb_model.fit(X_train, y_train)

print("Random Forest and XGBoost models trained successfully.")

Random Forest and XGBoost models trained successfully.


In [6]:
#Step 5: Evaluate the models

def evaluate_model(name, model, X_test, y_test):
    predictions = model.predict(X_test)
    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)
    
    print(f"--- {name} Evaluation ---")
    print(f"MAE:  {mae:.2f} minutes")
    print(f"RMSE: {rmse:.2f} minutes")
    print(f"R^2:  {r2:.4f}\n")

evaluate_model("Random Forest", rf_model, X_test, y_test)
evaluate_model("XGBoost", xgb_model, X_test, y_test)

# Select the Champion Model based on evaluation
champion_model = xgb_model

--- Random Forest Evaluation ---
MAE:  3.19 minutes
RMSE: 4.99 minutes
R^2:  0.9972

--- XGBoost Evaluation ---
MAE:  3.17 minutes
RMSE: 5.08 minutes
R^2:  0.9971



In [31]:
import numpy as np

def prepare_user_input(user_data):
    """
    Cleans and imputes missing web form data before feeding it to the ML model.
    """
    processed_data = user_data.copy()
    
    # 1. Handle Optional VO2 Max
    if processed_data.get('vo2_max') is None:
        # Use the physiological ratio formula if blank
        max_hr = processed_data['max_hr']
        rest_hr = processed_data['resting_hr']
        processed_data['vo2_max'] = 15.3 * (max_hr / rest_hr)
        
    # 2. Handle missing HM or FM Personal Bests (using 10K as a baseline)
    # If they haven't run a Half Marathon, estimate it: 10K time * 2.22
    if processed_data.get('pb_half_mins') is None and processed_data.get('pb_10k_mins') is not None:
        processed_data['pb_half_mins'] = processed_data['pb_10k_mins'] * 2.22
        
    # If they haven't run a Full Marathon, estimate it: Half Marathon time * 2.1
    if processed_data.get('pb_full_mins') is None and processed_data.get('pb_half_mins') is not None:
        processed_data['pb_full_mins'] = processed_data['pb_half_mins'] * 2.1
        
    return processed_data

def predict_race_time(model, user_data):
    """
    The main prediction function called by the web application.
    """
    # Clean the input first
    clean_data = prepare_user_input(user_data)
    
    input_df = pd.DataFrame([clean_data])
    input_df = input_df[features] # Align columns
    
    predicted_minutes = model.predict(input_df)[0]
    
    hours = int(predicted_minutes // 60)
    minutes = int(predicted_minutes % 60)
    seconds = int((predicted_minutes * 60) % 60)
    
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"



In [39]:
# --- Testing the Web App Scenario ---
# A recreational runner who doesn't know their VO2 Max and has NEVER run a HM or FM
web_form_submission = {
    'age': 24, 
    'gender_encoded': 0, 
    'resting_hr': 47, 
    'max_hr': 208, 
    'vo2_max': 50,             # Optional: Left blank!
    'weekly_mileage_km': 65.0, 
    'avg_easy_pace_min_km': 6.50, 
    'avg_tempo_pace_min_km': 5.40, 
    'training_consistency': 0.70, 
    'pb_5k_mins': 25.45, 
    'pb_10k_mins': 55.0, 
    'pb_half_mins': None,        # Optional: Never run one!
    'pb_full_mins': None,        # Optional: Never run one!
    'target_distance_km': 21.0975 # They are training for their first Half Marathon #21.0975
}

pred_time = predict_race_time(champion_model, web_form_submission)
print(f"Predicted First Half Marathon Time: {pred_time}")

Predicted First Half Marathon Time: 02:06:05
